In [ ]:
from google.colab import userdata
import os, subprocess
repo = "https://github.com/hh4832/target_price_trading.git"
root = "/content/target_price_trading"
if not os.path.isdir(os.path.join(root, ".git")):
    subprocess.run(["git", "clone", repo, root], check=True)
os.chdir(root)
subprocess.run(["git", "remote", "-v"], check=True)
subprocess.run(["git", "status", "--short"], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
from google.colab import drive
# Mount is optional for inspection; the pipeline writes by verified folder ID through Drive API.
drive.mount("/content/drive")
for key in ("GOOGLE_SERVICE_ACCOUNT_JSON", "FINLAB_REFRESH_TOKEN", "FINLAB_SESSION_ID", "FINLAB_API_KEY"):
    value = userdata.get(key)
    if value:
        os.environ[key] = value
if not all(os.environ.get(k) for k in ("FINLAB_REFRESH_TOKEN", "FINLAB_SESSION_ID", "FINLAB_API_KEY")):
    import finlab
    finlab.login()
if not os.environ.get("GOOGLE_SERVICE_ACCOUNT_JSON"):
    raise RuntimeError("Add GOOGLE_SERVICE_ACCOUNT_JSON to Colab Secrets and share the specified folder and Sheet with its client_email")


In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([sys.executable, "-m", "src.pipeline"], check=True)


In [ ]:
from src.config import FOLDER_ID
from src.output import clients, find_child, read_csv, read_state, verify_folder
drive_client, _ = clients()
verify_folder(drive_client, FOLDER_ID)
state = read_state(drive_client, FOLDER_ID)
print("Archive:", state.get("last_archive", "DATA_NOT_UPDATED / no previous archive"))
if state.get("last_archive"):
    archive = find_child(drive_client, FOLDER_ID, state["last_archive"])
    from io import BytesIO
    import pandas as pd
    item = find_child(drive_client, archive["id"], "daily_screen.csv")
    display(pd.read_csv(BytesIO(drive_client.files().get_media(fileId=item["id"]).execute())).head(20))
